# 获取结构化结果方式

## 1、使用with_structured_output

举例：

In [5]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 1、读取.env配置文件中的信息。相关的环境变量以.env文件中的优先
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 2、模型的初始化
llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    }

)

In [6]:
from pydantic import BaseModel, Field
from rich import print as rprint

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分（10分制）")


structured_model = llm_deepseek.with_structured_output(Movie,include_raw=True)
response = structured_model.invoke("帮我介绍一下星际穿越这个电影")

print(type(response))
rprint(response)   # 输出结果中就会包含原始的AIMessage

<class 'dict'>


{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 89,
                'prompt_tokens': 339,
                'total_tokens': 428,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                'prompt_cache_hit_tokens': 0,
                'prompt_cache_miss_tokens': 339
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-flash',
            'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
            'id': 'c7c9f31a-f9fb-48f9-ade8-31b85835b263',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--01a09440-59fd-7bb3-b799-ba21b8479ec3-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'rating': 9.4},
                'id': 'call_00_vNiWBarn893PR5y3EIm05316',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 339,
            'output_tokens': 89,
            'total_tokens': 428,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {}
        }
    ),
    'parsed': Movie(title='星际穿越', year=2014, director='克里斯托弗·诺兰', rating=9.4),
    'parsing_error': None
}

## 2、使用输出解析器（了解）

In [8]:

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1. 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system","回答用户问题,必须始终输出一个包含title(电影标题)和year(上映年份)的 JSON 对象"),
    ("human","问题：{question}")
])

# 2. 模型初始化
# 从.env文件中加载环境变量
load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)

# 5. 创建链
chain = prompt_template | model | parser

# 6. 调用（返回字典）
# response = chain.invoke({"question": "介绍电影《盗梦空间》"})

response = parser.invoke(model.invoke(prompt_template.invoke({"question": "介绍电影《盗梦空间》"})))


print(type(response))
print(response)

<class 'dict'>
{'title': '盗梦空间', 'year': 2010}
